# Statistical and Machine Learning
# Homework 1 : Introduction and Learnability

## Learning problem
* **Input space :** $\mathcal{X} = [0, 1]$ muni d'une distribution uniforme
* **Target function :** unknown threshold classifier $f_\theta(x) = \mathbb{1}\{x > \theta\}$, où $\theta \in (0, 1)$ est fixé.
* **Labels :** $y = f_\theta(x)$
* **Agnostic setting :** si on est dans APAC, alors les étiquettes peuvent être inversées avec la probabilité de $\eta$.

## 1. Data Generation
Dans cette question, on génère des training datasets $S = \{(x_i, y_i)\}_{i=1}^m$ de différentes tailles $m \in \{5, 10, 20, 50, 100, 200\}$.
On choisit la limite à $\theta = 0.4$. Si on est dans le cadre PAC où on a l'hypothèse de réalisabilité (Realizability Assumption) alors $\eta = 0$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Fixer le point de départ (seed) pour reproduire les expériences et avoir les mêmes résultats pour les interprétations
np.random.seed(0)

# Paramètres globaux de l'expérience
THETA = 0.4  # Limite \theta \in (0, 1)
m_tab = [5, 10, 20, 50, 100, 200]  # tailles des échantillons

def fonction_cible(x, theta=THETA):
  """Calcule le résultat de la fonction cible f_theta(x).
  Renvoie 1 si x > theta, 0 sinon.
  """
  res = np.where(x > theta, 1, 0)
  return res

def generer_dataset(m, theta=THETA, eta=0.0):
  """Génère un training dataset S de taille m.
  Entrées :
  - m (int) : Nombre d'exemples dans le training set
  - theta (float) : la limite theta dans (0, 1).
  - eta (float) : la probabilité que l'étiquette soit inversée dans le cadre APAC :
                  eta = 0.0 correspond au cadre PAC.
                  eta > 0.0 correspond au cadre Agnostic PAC.
  Retourne :
  - x_tab (np.ndarray) : Tableau 1D d'instances x tirées uniformément dans [0, 1].
  - y_tab (np.ndarray) : Tableau 1D d'étiquettes y dans {0, 1} associés aux instances x de x_tab.
  """
  # 1. Tirage uniforme des instances x dans X = [0, 1] m fois
  x_tab = np.random.uniform(0.0, 1.0, size=m)

  # 2. Génération des étiquettes selon la fonction cible y = f_theta(x)
  y_tab = fonction_cible(x_tab, theta)

  # 3. Lorsque APAC donc lorsqu'on n'a pas l'hypothèse de réalisabilité : inversion indépendante de chaque étiquette avec probabilité eta
  if eta > 0.0:
    inverser_tab = np.random.uniform(0.0, 1.0, size=m) < eta #Génère aléatoirement selon eta le tableau qui contient true s'il faut inverser l'étiquette de l'instance x donnée, false sinon"
    y_tab = np.where(inverser_tab, 1 - y_tab, y_tab) #Modifie les étiquettes associées aux instances selon le tableau inverser_tab"
  return x_tab, y_tab

# Génération des datasets pour toutes les tailles m dans le cadre PAC (avec l'hypothèse de réalisabilité) : eta = 0.0
  datasets = {}
  for m in m_tab:
    datasets[m] = generer_dataset(m, theta=THETA, eta=0.0)

print(
    "Training datasets générés avec succès pour m appartenant à :",
    list(datasets.keys()),
)

In [ ]:
# Visualisation des training set générés pour les différentes tailles m
fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharey=True)
axes = axes.flatten()

x_grille = np.linspace(0, 1, 500)
y_courbe = fonction_cible(x_grille, THETA)

for idx, m in enumerate(m_tab):
  ax = axes[idx]
  x_ech, y_ech = datasets[m]

  # Tracé de la fonction cible en escalier f_theta(x)
  ax.plot(
      x_grille,
      y_courbe,
      color='black',
      linestyle='--',
      alpha=0.4,
      label=r'Fonction cible $f_\theta(x)$',
  )

  # Tracé de la ligne verticale représentant la limite theta
  ax.axvline(
      x=THETA,
      color='red',
      linestyle=':',
      linewidth=1.5,
      label=rf'Limite $\theta={THETA}$',
  )

  # Séparation des classes pour une visualisation claire
  x_classe0 = x_ech[y_ech == 0]
  x_classe1 = x_ech[y_ech == 1]

  ax.scatter(
      x_classe0,
      np.zeros_like(x_classe0),
      color='tab:blue',
      marker='o',
      s=50,
      alpha=0.8,
      edgecolors='k',
      label=r'Classe 0 ($y=0$)',
  )
  ax.scatter(
      x_classe1,
      np.ones_like(x_classe1),
      color='tab:orange',
      marker='^',
      s=60,
      alpha=0.8,
      edgecolors='k',
      label=r'Classe 1($y=1$)',
  )

  ax.set_title(
      rf"$m = {m}$", fontsize=12, fontweight='bold'
  )
  ax.set_xlabel(r'$x \in [0, 1]$')
  if idx % 3 == 0:
    ax.set_ylabel(r'Étiquette $y \in \{0, 1\}$')
  ax.set_yticks([0, 1])
  ax.set_xlim(-0.05, 1.05)
  ax.set_ylim(-0.2, 1.2)
  ax.grid(True, linestyle='--', alpha=0.5)

  if idx == 0:
    ax.legend(loc='upper left', fontsize=9)

plt.suptitle(
    r"PAC Learning ($\eta = 0$) : Training sets pour"
    r' différentes tailles $m$',
    fontsize=14,
    fontweight='bold',
)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

### Interprétation de la génération de training datasets
1. **PAC Learning ($\eta = 0$) :**
   * Tous les exemples générés respectent bien la fonction cible $f_\theta(x)$ puisqu'on est dans le PAC Learning et que donc on suppose que l'hypothèse de réalisabilité est respectée, c'est-à-dire qu'il existe bien une hypothèse dans la classe d'hypothèses qui donne les mêmes résultats que la fonction cible.
   * Chaque point situé à gauche de la ligne verticale rouge ($x \le 0.4$) possède l'étiquette $y=0$ et chaque point situé à droite ($x > 0.4$) possède l'étiquette $y=1$ par application de la fonction cible.
   Les deux classes sont donc parfaitement séparables puisqu'il n'y a pas d'erreur possible lors de la détermination de l'étiquette parce qu'on est dans le PAC Learning et que donc la fonction cible est respectée.
   * On est donc dans le "Helpful Teacher" comme type d'apprentissage, c'est-à-dire que les training datasets ont été généré pour aider, contrairement au "Adversarial Teacher" où on aurait eu des jeux de données avec des erreurs (dans le contexte APAC avec $\eta > 0$).
3. **Importance de la taille d'échantillon $m$ :** 
   * On constate que pour des petites tailles d'échantillon ($m = 5$, $m = 10$ et $m = 15$) il y a moins de points bleus que de triangles oranges, c'est-à-dire qu'on a moins d'étiquettes $y=0$ que d'étiquettes $y=1$ ce qui est un résultat qui semble logique. En effet, la limite choisie étant $0.4$ qui est strictement inférieure au milieu $0.5$ (x étant généré entre 0 et 1), cela semble donc normal qu'on ait plus d'étiquettes $y=1$ que d'étiquettes $y=0$.